# Related Papers:
- https://aclanthology.org/2024.acl-short.2.pdf
- https://aclanthology.org/2024.emnlp-main.171.pdf
- https://arxiv.org/pdf/2408.09174v1
- https://aclanthology.org/2025.findings-emnlp.638.pdf
- https://aclanthology.org/2025.emnlp-main.363.pdf

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import random
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

In [2]:
import json
import numpy as np
import pandas as pd
from collections import Counter

In [3]:
def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(file_name, file):
    with open(os.path.join(REPO_PATH, "data_files", f"{file_name}.json"), "w", encoding="utf-8") as out_file:
        json.dump(file, out_file, indent=4)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def save_jsonl(file_name, records):
    path = os.path.join(REPO_PATH, "data_files", f"{file_name}.jsonl")
    with open(path, "w", encoding="utf-8") as out_file:
        for record in records:
            out_file.write(json.dumps(record, ensure_ascii=False) + "\n")

# Data Creation:

## Harvest Wikipedia pages with tables:

In [91]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester()
# Run Phase 1 - Harvesting titles based on keywords
harvester.run_complete_harvest()
# Run Phase 2 - Filtering pages for tables
await harvester.filter_pages_async()

Harvesting Pages By Keywords:   0%|          | 0/233 [00:00<?, ?keyword/s]


TOTAL UNIQUE TITLES COLLECTED: 167913



Method,Keyword,Count
Prefix,List of cities,1537
Prefix,List of countries,1278
Prefix,List of towns,501
Prefix,List of villages,483
Prefix,List of districts,375
Prefix,List of municipalities,1021
Prefix,List of communes,194
Prefix,List of counties,259
Prefix,List of provinces,84
Prefix,List of territories,22


Filtering Pages with Tables:   0%|          | 0/167913 [00:00<?, ?page/s]


TOTAL PAGES WITH TABLES COLLECTED: 62306



In [93]:
save_json("table_titles", harvester.valid_titles)

In [174]:
table_titles = load_json("C:/Users/Daniel/Desktop/KnowledgeMaze/data_files/table_titles.json")

## Parse and filter Wikipedia tables:

In [271]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
validated_tables_file_name = await extractor.run_with_progress(table_titles)
validated_tables = load_jsonl(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
cleaned_data = processor.process()

Pre-processing: cluster similar pages and discard similarities ---> 6426 pages discarded.


Extracting Tables:   0%|          | 0/55880 [00:00<?, ?Page/s]

Starting Extraction: 55880 titles | Async Limit: 20 | CPU Workers: 19
TOTAL PAGES WITH VALID TABLES: 32076 OUT OF 55880 (57.4%)


Filtering & Cleaning:   0%|          | 0/94101 [00:00<?, ?it/s]


════════════════════════════════════════════
📊 TABLE PROCESSING SUMMARY (Limit: 12)
════════════════════════════════════════════
Total tables scanned:      94101
Dropped (Bad Headers):     8435
Dropped (Cell too long):   2841
Dropped (Repeat Rows):     1783
Short tables found:        39837
Unique source pages:       16940
Final sampled (1/page):    16940
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE TABLES ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

,Rank,Player,Career,Caps,Goals
0,1,Christine Sinclair,2000–2023,331,190
1,2,Sophie Schmidt,2005–2023,226,20
2,3,Diana Matheson,2003–2021,206,19
3,4,Desiree Scott,2010–2025,188,0
4,5,Rhian Wilkinson,2003–2017,181,7
5,6,Kadeisha Buchanan,2013–present,154,6
6,7,Ashley Lawrence,2013–present,151,8
7,7,Jessie Fleming,2013–present,151,20
8,9,Brittany Baxter,2002–2014,132,5
9,9,Andrea Neil,1991–2007,132,24



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Manchester City F.C. 3–2 Queens Park Rangers F.C. > Match > Statistics
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Statistic,Manchester City,Queens Park Rangers
0,Goals scored,3,2
1,Possession,81.3%,18.7%
2,Shots on target,15,3
3,Shots off target,10,0
4,Blocked shots,19,0
5,Corner kicks,19,0
6,Fouls,4,7
7,Tackles,15,22
8,Offsides,1,1
9,Yellow cards,2,1



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Swinburne University of Technology > Notable people > Academics and staff
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Member,Executive team,Period
0,Pascale Quester,Vice-Chancellor,2020–present
1,Chris Pilgrim,Pro Vice-Chancellor (Academic),
2,Tara Magdalinski,Pro Vice-Chancellor (Education and Quality),
3,Sarah Maddison,Pro Vice-Chancellor (Academic Innovation and C...,
4,Bronwyn Fox,Deputy Vice-Chancellor (Research and Enterprise),
5,Beth Webster,Pro Vice-Chancellor (Research Policy and Impact),
6,Qing-Lon Han,Pro Vice-Chancellor (Research Quality),
7,Alan Kin-Tak Lau,Pro Vice-Chancellor (International Research De...,
8,Blair Kuys,Dean of the School of Design and Architecture,
9,Matthew Bailes,Director of the Data Science Research Institut...,


## Create Potential Entity Groups:

In [296]:
list_of_cleaned_data = [a for a in cleaned_data if a['table_name'].lower().startswith('list of')]
len(list_of_cleaned_data)

5690

In [297]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(random.sample(list_of_cleaned_data, 1000))
await group_constructor.get_groups()

🚀 Grouping 1000 tables using Entity-Focus logic...


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/1000 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════════════════════════
                               📊 GROUPING REPORT                                
════════════════════════════════════════════════════════════════════════════════
  ✅ Valid Groups:   933
  ❌ Invalid Tables: 67
════════════════════════════════════════════════════════════════════════════════

✨ SAMPLE VALID NAMES:
  → Towns on the Isle of Wight
  → Senior Staff Officers of the Indian Air Force
  → Railway stations in India starting with O

────────────────────────────────────────────────────────────────────────────────
                     🕵️ REJECTION DEBUGGER (Random Sample)                      
────────────────────────────────────────────────────────────────────────────────

[TABLE]: List of founders of religious traditions > Bibliography
[REASON]: The table contains categories of religious traditions rather than unique instances of entities. It does not meet the criteria for a coherent group of entities

In [310]:
stst = []
for sample in random.sample(group_constructor.valid_tables, 100):
    try:
        stst.append(list(sample['Table data'][0].keys()).index(sample['Entity column']))
        print(sample['Group name'])
        print(sample['Entity column'])
        for row in sample['Table data'][:3]:
            print(row)
        print("=" * 100)
        # display(pd.DataFrame(sample['Table data']))
    except:
        pass
        stst.append('Failed')
# print(Counter(stst))

Grammy Awards received by Carly Simon
Award
{'Year': 1974, 'Work': 'No Secrets', 'Award': 'Best Engineered Recording', 'Recipient': 'Robin Geoffrey Cable and Bill Schnee', 'Result': 'Nominated'}
{'Year': 1976, 'Work': 'Playing Possum', 'Award': 'Best Album Package', 'Recipient': 'Gene Christensen A', 'Result': 'Nominated'}
{'Year': 1978, 'Work': '" Nobody Does It Better "', 'Award': 'Song of the Year', 'Recipient': 'Marvin Hamlisch and Carole Bayer Sager', 'Result': 'Nominated'}
Massacres in the Ottoman Empire before 1914
Name
{'Name': 'Massacres during the Greek War of Independence', 'Date': '1821–1829', 'Location': 'Ottoman Empire', 'Deaths': 'Unknown', 'Responsible Party': 'Ottoman government and Greek rebels', 'Victims': 'Greeks , Turks, Albanians and Jews', 'Notes': 'Massacres were committed by both sides during the conflict.'}
{'Name': 'Massacres of Badr Khan', 'Date': '1843-1846', 'Location': 'Hakkari', 'Deaths': '4,000-7,000', 'Responsible Party': 'Kurdish Emirs of Bhutan, Badr

In [663]:
for sample in random.sample(group_constructor.valid_tables, 25):
    print(f"Group name: {sample['Group name']}")
    print(f"Wikipedia page: {sample['Wikipedia link']}")
    print(f"Entity column: {sample['Entity column']}")
    display(pd.DataFrame(sample['Table data']))
    print()
    print()

Group name: Artists with multiple nominations and awards at the Brit Awards 2006
Wikipedia page: https://en.wikipedia.org/wiki/Brit_Awards_2006#Multiple_nominations_and_awards
Entity column: Artist


,Awards,Artist
0,3,Kaiser Chiefs
1,2 (3),Coldplay
2,2 (3),Green Day
3,2 (3),James Blunt




Group name: Awards and nominations received by Sam Smith at the iHeartRadio Music Awards
Wikipedia page: https://en.wikipedia.org/wiki/List_of_awards_and_nominations_received_by_Sam_Smith#iHeartRadio_Music_Awards
Entity column: Nominee / work


,Year,Nominee / work,Award,Result
0,2015,Sam Smith,Artist of the Year,Nominated
1,2015,Sam Smith,Best New Artist,Won
2,2015,"""Stay with Me""",Song of the Year,Nominated
3,2015,"""Stay with Me""",Best Lyrics,Nominated
4,2016,Sam Smith,Male Artist of the Year,Nominated
5,2016,In the Lonely Hour,Album of the Year,Nominated
6,2016,"""Writing's on the Wall""",Best Song from a Movie,Nominated
7,2016,""" Hotline Bling """,Best Cover Song,Nominated
8,2020,"""Dancing With A Stranger"" (with Normani)",Best Collaboration,Nominated
9,2020,"""Dancing With A Stranger"" (with Normani)",Best Music Video,Nominated




Group name: 1965 Tennessee Volunteers football schedule
Wikipedia page: https://en.wikipedia.org/wiki/1965_Tennessee_Volunteers_football_team#Schedule
Entity column: Opponent


...

,Race / Ethnicity ( NH = Non-Hispanic ),Pop 2000,Pop 2010,Pop 2020,% 2000,% 2010,% 2020
0,White (NH),122708,133790,128348,82.79%,79.24%,73.25%
1,Black or African American (NH),15526,17844,17549,10.48%,10.57%,10.02%
2,Native American or Alaska Native (NH),1096,1033,795,0.74%,0.61%,0.45%
3,Asian (NH),2534,3298,4068,1.71%,1.95%,2.32%
4,Native Hawaiian or Pacific Islander alone (NH),111,140,194,0.07%,0.08%,0.11%
5,Other race alone (NH),164,223,720,0.11%,0.13%,0.41%
6,Mixed race or Multiracial (NH),2487,4417,9696,1.68%,2.62%,5.53%
7,Hispanic or Latino,3591,8107,13846,2.42%,4.80%,7.90%




Group name: Guns N' Roses touring members
Wikipedia page: https://en.wikipedia.org/wiki/List_of_Guns_N'_Roses_members#Members
Entity column: Name


,Name,Years active,Instruments,Details
0,Fred Coury,1987–1988,drums,Coury substituted for Steven Adler between lat...
1,"Kid ""Haggis"" Chaos (Stephen Harris)",1988,bass,Harris replaced Duff McKagan for a show on May...
2,Don Henley,1989,drums backing vocals,Henley replaced Steven Adler for the 1989 Amer...
3,Teddy Andreadis,1991–1993,keyboards percussion harmonica backing vocals,Additional members of the Use Your Illusion To...
4,Roberta Freeman,1991–1993,backing vocals tambourine,Additional members of the Use Your Illusion To...
5,Tracey Amos,1991–1993,backing vocals tambourine,Additional members of the Use Your Illusion To...
6,Diane Jones,1991–1993,backing vocals tambourine,Additional members of the Use Your Illusion To...
7,Cece Worrall,1991–1993,saxophones,Additional members of the Use Your Illusion To...
8,Lisa Maxwell,1991–1993,saxophones,Additional members of the Use Your Illusion To...
9,Anne King,1991–1993,trumpet,Additional members of the Use Your Illusion To...




Group name: Racial and ethnic composition of Rapids City, Illinois
Wikipedia page: https://en.wikipedia.org/wiki/Rapids_City,_Illinois#Demographics
Entity column: Race / Ethnicity ( NH = Non-Hispanic )


,Race / Ethnicity ( NH = Non-Hispanic ),Pop 2000,Pop 2010,Pop 2020,% 2000,% 2010,% 2020
0,White alone (NH),924,922,889,96.96%,96.14%,92.22%
1,Black or African American alone (NH),0,2,8,0.00%,0.21%,0.83%
2,Native American or Alaska Native alone (NH),0,1,1,0.00%,0.10%,0.10%
3,Asian alone (NH),1,3,1,0.10%,0.31%,0.10%
4,Native Hawaiian or Pacific Islander alone (NH),1,0,0,0.10%,0.00%,0.00%
5,Other race alone (NH),1,0,1,0.10%,0.00%,0.10%
6,Mixed race or Multiracial (NH),6,10,27,0.63%,1.04%,2.80%
7,Hispanic or Latino (any race),20,21,37,2.10%,2.19%,3.84%


## Create FanOut-style Questions out of the Entity Groups:

### Rule-Based Strategy Candidate Selection:

In [311]:
from knowledge_maze.create_dataset.create_questions.question_generation import StrategySelector

selector = StrategySelector(group_constructor.valid_tables)
selector.get_strategies()

Sanitizing tables:   0%|          | 0/933 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/933 [00:00<?, ?it/s]


✅ Strategy Selection Complete.

═══════════════════════════════════════════════════════
📊 STRATEGY SELECTION SUMMARY
-------------------------------------------------------
✅ Groups Passed:     897
❌ Groups Discarded:  36
📈 Success Rate:      96.1%
-------------------------------------------------------
Strategy Distribution (Top 4 per table):
  - UNIQUE_DIVERSITY          : 657
  - COUNT_TOTAL               : 473
  - EXTREME_TEMPORAL          : 472
  - FILTER_COUNT_TEMPORAL     : 354
  - FREQUENCY_MODE            : 266
  - CATEGORICAL_FILTER_LOOKUP : 235
  - EXTREME_NUMERICAL         : 91
  - AGGREGATION               : 84
  - TOP_K_RANKING             : 83
  - COMPARATIVE_DIFFERENCE    : 57
  - FILTER_COUNT_NUMERICAL    : 34
-------------------------------------------------------
POSSIBLE STRATEGIES  | GROUP COUNT  | PERCENTAGE
-------------------------------------------------------
#1                   | 79           |       8.8%
#2                   | 182          |      20.3%
#3 

In [316]:
selector.debug_strategy_samples("FREQUENCY_MODE")

Found 266 total tables with this strategy. Showing 5 random samples.



🔗 Source: https://en.wikipedia.org/wiki/List_of_motor_vehicle_awards#People


,Award,Sponsor
0,Automotive Executive of the Year Award,DNV GL - Business Assurance
1,Automotive Hall of Fame,MotorCities National Heritage Area
2,Car Designer of the Century,Global Automotive Elections Foundation
3,Car Engineer of the Century,Global Automotive Elections Foundation
4,Car Entrepreneur of the Century,Global Automotive Elections Foundation
5,Louis Schwitzer Award,Society of Automotive Engineers
6,Time Magazine Quality Dealer Award,"Time , Ally Financial"


🔗 Source: https://en.wikipedia.org/wiki/List_of_new_members_of_the_109th_United_States_Congress#Senate


,Senator,Prior background
0,Ken Salazar (D),Attorney General of Colorado
1,Mel Martinez (R),U.S. Secretary of Housing and Urban Developmen...
2,Johnny Isakson (R),U.S. House of Representatives Georgia Senate ...
3,Barack Obama (D),Illinois Senate
4,David Vitter (R),U.S. House of Representatives Louisiana House...
5,Richard Burr (R),U.S. House of Representatives
6,Tom Coburn (R),U.S. House of Representatives
7,Jim DeMint (R),U.S. House of Representatives
8,John Thune (R),U.S. House of Representatives


...

,Nominated work,Category
0,August: Osage County,Outstanding Cast in a Motion Picture
1,12 Years a Slave,Outstanding Cast in a Motion Picture
2,The Imitation Game,Outstanding Cast in a Motion Picture
3,The Imitation Game,Outstanding Male Actor in a Leading Role
4,Sherlock,Outstanding Male Actor in a Miniseries or Movie
5,Sherlock,Outstanding Male Actor in a Miniseries or Movie
6,The Power of the Dog,Outstanding Male Actor in a Leading Role


🔗 Source: https://en.wikipedia.org/wiki/List_of_power_stations_in_Idaho#Renewable_power_stations


,Name,Primary fuel
0,Bannock County Landfill Gas,Landfill gas
1,Cargill Dry Creek Biofactory,Biogas
2,Clearwater Paper Biomass,Wood/wood waste
3,Fighting Creek Landfill Gas,Landfill gas
4,Hidden Hollow Energy,Landfill gas
5,Milner Butte LFGE,Landfill gas
6,Plummer Cogen,Wood/wood waste
7,Rock Creek Dairy,Biogas
8,Simplot Don Plant,Industrial waste heat
9,Tamarack Energy,Wood/wood waste


🔗 Source: https://en.wikipedia.org/wiki/List_of_Dungeons_&_Dragons_video_games#List_of_games


,Title,Setting
0,Dungeons & Dragons Computer Fantasy Game,Generic
1,Advanced Dungeons & Dragons: Cloudy Mountain,Generic
2,Advanced Dungeons & Dragons: Treasure of Tarmin,Generic
3,Spelljammer: Pirates of Realmspace,Spelljammer
4,Stronghold,Generic
5,Al-Qadim: The Genie's Curse,Al-Qadim
6,Slayer,Generic
7,DeathKeep,Generic
8,Birthright: The Gorgon's Alliance,Birthright
9,Dungeons & Dragons Tactics,Generic


### Model-Based Question Generation:

In [312]:
from knowledge_maze.create_dataset.create_questions.question_generation import TableSanitizer, StrategySelector, QuestionGenerator

generator = QuestionGenerator(random.sample(selector.updated_data, 100),
                              strategy_definitions=load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await generator.generate_questions()

🚀 Starting Question Generation for 100 tables...
--- Round 1: Processing 100 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/100 [00:00<?, ?it/s]

--- Round 2: Processing 1 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/1 [00:00<?, ?it/s]


═════════════════════════════════════════════════════════════════
                  📊 QUESTION GENERATION SUMMARY                  
═════════════════════════════════════════════════════════════════
Initial Tables:      100
Successfully Gen:    100
Exhausted Strats:    0
Success Rate:        100.0%
-----------------------------------------------------------------
STRATEGY                     | SUCCESS  | TRIED    | RATE
-----------------------------------------------------------------
COUNT_TOTAL                  | 16       | 16       | 100%
UNIQUE_DIVERSITY             | 15       | 15       | 100%
EXTREME_TEMPORAL             | 14       | 14       | 100%
FILTER_COUNT_TEMPORAL        | 13       | 13       | 100%
FREQUENCY_MODE               | 13       | 14       |  93%
CATEGORICAL_FILTER_LOOKUP    | 13       | 13       | 100%
AGGREGATION                  | 4        | 4        | 100%
EXTREME_NUMERICAL            | 3        | 3        | 100%
TOP_K_RANKING                | 3        | 3   

,Award,Sponsor
0,Automotive Executive of the Year Award,DNV GL - Business Assurance
1,Automotive Hall of Fame,MotorCities National Heritage Area
2,Car Designer of the Century,Global Automotive Elections Foundation
3,Car Engineer of the Century,Global Automotive Elections Foundation
4,Car Entrepreneur of the Century,Global Automotive Elections Foundation
5,Louis Schwitzer Award,Society of Automotive Engineers
6,Time Magazine Quality Dealer Award,"Time , Ally Financial"


-----------------------------------------------------------------


## Question Validation:

### Model-Based Question Validator (Gemini 2.5 Flash):

In [313]:
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator
validator = QuestionValidator(generator.valid_results, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await validator.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/91 [00:00<?, ?it/s]

Network
Lifetime
NBC
NBC
ESPN
NBC


Airline,2024
United Airlines,311
American Airlines,293
Delta Air Lines,288
Southwest Airlines,178
Air Canada,104
JetBlue,66
Alaska Airlines,64
Spirit Airlines,53
Frontier Airlines,40


Name,Years active
Steve Harris,1975–present
Dave Murray,1976–1977 1978–present
Adrian Smith,1980–1990 1999–present
Bruce Dickinson,1981–1993 1999–present
Nicko McBrain,1982–present (inactive from touring as of 2024)
Janick Gers,1990–present


Title,Month
Pathaan,January
Chhaava,February
The Kashmir Files,March
Kisi Ka Bhai Kisi Ki Jaan,April
Yeh Jawaani Hai Deewani,May
Sanju,June
Bajrangi Bhaijaan,July
Stree 2,August
Jawan,September
Secret Superstar,October


Name,ITIS TSN
Bombus alpinus,714857
Bombus balteatus,714786
Bombus hyperboreus,714811
Bombus kirbiellus,1128121
Bombus kluanensis,1129722
High Arctic Bumble Bee,1129726
Bombus neoboreus,714825
Bombus polaris,714831
Bombus pyrrhopygus,1129727


Title,Year
Cléopâtre,1910
Cleopatra,1912
Antony and Cleopatra,1913
Cleopatra,1917


Coach,Coach of the Year
Bruce Arena,"4 (1997, 2009, 2011, 2021)"
Bob Bradley,"3 (1998, 2006, 2019)"
Sigi Schmid,"2 (1999, 2008)"
Frank Yallop,"2 (2001, 2012)"
Jim Curtin,"2 (2020, 2022)"
Dominic Kinnear,1 (2005)
Greg Vanney,1 (2017)
Caleb Porter,1 (2013)
Thomas Rongen,1 (1996)
Brian Schmetzer,—


### Manual Validation:

In [318]:
tagger = ManualTagger(validator.approved_data, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
tagger.display() # Render the output area

📦 Progress Loaded: 14 samples already tagged in 'tagger_progress.json'.


Output()

# Tests:

## 1. Harvest Wikipedia pages with tables:

1. **The Discovery Phase (The Wide Net)**
   - Input: 22 Subject Groups (e.g., Infrastructure, Sports).
   - Action: We hit the Wikipedia API with two distinct patterns:
     - Prefix Mining: Grabbing every page starting with "Entity in/of..."
     - Search Mining: Finding every page with "List of [Entity]" in the title.
     - Goal: Maximum recall. We want to find everything that might be a list.
2. **The Validation Phase (The Filter)**
   - Action: We open every page found in Phase 1 (asynchronously for speed).
   - Logic: We look for the "DNA" of a table: the {| or wikitable tags.
   - Outcome: We discard the "stub" articles or prose-only pages that don't actually contain structured data.
3. **The Clustering Phase (The De-duplicator)**
   - Problem: Wikipedia often has "List of 2023 Films" and "List of films of 2023." These are duplicates.
   - Action: Using TF-IDF to cluster similar page titles.
   - If two titles are >80% similar, we draw a line (edge) between them.
   - Pick a representative from each cluster to continue.

In [12]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiMiner

miner = WikiMiner()
await miner.extract()

Mining Pages:   0%|          | 0/320 [00:00<?, ?entity/s]

Filtering Pages with Tables:   0%|          | 0/89666 [00:00<?, ?page/s]

Computing Similarities for 43937 Titles:   0%|          | 0/43937 [00:00<?, ?titles/s]

Picking Representatives from 39391 Clusters:   0%|          | 0/39391 [00:00<?, ?cluster/s]

## 2. Parse and filter valid Wikipedia tables:

1. **Geometric Fix** - Rebuilds the "hidden grid" of the table. It maps merged cells (rowspans) to their correct spots so the data doesn't shift.
2. **Context Grab** - Captures the Section Path (e.g., Earthquakes > 1954) and the Intro Sentence above the table to explain what the data means.
3. **Trash Filter** - Deletes "Legend" tables, empty tables, and tiny snippets. If it's less than 4 rows or 50% empty, it’s gone.
4. **Variety Control** - "Detects ""Clones"" (like 12 identical monthly tables). It keeps a maximum of 4 different tables per page to ensure the dataset isn't boring.

In [93]:
save_json("valid_titles", miner.valid_titles)

In [4]:
valid_titles = load_json(os.path.join(REPO_PATH, "data_files/valid_titles.json"))

In [5]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/candidate_tables.jsonl"))
validated_tables_file_name = await extractor.run_with_progress(valid_titles) # (miner.valid_titles)

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/candidate_tables.jsonl"))
cleaned_data = processor.process()

Extracting Tables:   0%|          | 0/39391 [00:00<?, ?Page/s]

Starting Extraction: 39391 titles | Async Limit: 5 | CPU Workers: 7


Filtering & Cleaning:   0%|          | 0/64646 [00:00<?, ?it/s]

Legend / Key Filter,553
Header Parsing Failures,"3,883"
Cell Content Too Long,"1,935"
Malformed Row Structures,"1,604"
Alphabetical Lists Removed,622
Tables with no Headers,585
Long Tables Removed,"31,673"
Intra-Page Duplicates Removed,"8,878"


In [6]:
processor.display_sample()

,Formation name,Created,Ceased to exist,Locations served,Notes
0,1st Division,18 June 1809,April 1814,"Portugal, Spain, France","At the conclusion of the Peninsular War, in 18..."
1,2nd Division,18 June 1809,April 1814,"Portugal, Spain, France","At the conclusion of the Peninsular War, in 18..."
2,3rd Division,18 June 1809,April 1814,"Portugal, Spain, France","At the conclusion of the Peninsular War, in 18..."
3,4th Division,18 June 1809,April 1814,"Portugal, Spain, France","At the conclusion of the Peninsular War, in 18..."
4,5th Division,6 October 1810,April 1814,"Portugal, Spain, France","At the conclusion of the Peninsular War, in 18..."
5,6th Division,6 October 1810,April 1814,"Portugal, Spain, France","At the conclusion of the Peninsular War, in 18..."
6,7th Division,5 March 1811,April 1814,"Portugal, Spain, France","At the conclusion of the Peninsular War, in 18..."
7,Light Division,22 February 1810,1814?,,
8,1st Cavalry Division,19 June 1811,21 April 1813,Spain,Served with main field army. Broken up into in...
9,2nd Cavalry Division,19 June 1811,21 April 1813,Spain,Served with Hill's Corps. Broken up into indiv...


## 3. Create Potential Entity Groups:

A table is considered a valid **entity group** if it satisfies the following conditions:

- **Entity Column:** The table must contain a column where each cell is a unique instance of the same entity type (e.g., person, film, city, team).
- **Subject Over Data:** The group name must describe **what the rows are**, not attributes or statistics about them.
- **Entity–Name Alignment:** The group name must accurately match the type of entities in the entity column.
- **No Repeated Labels:** The entity column cannot contain generic labels or repeated categories (e.g., "Height", "Weight").
- **No Summary Tables:** Tables that summarize metrics (e.g., "Total Population", "Revenue") rather than listing entities are invalid.
- **Clear Grouping Rule:** The group must explain why these entities appear together (e.g., *Top 10 films by revenue*, *Citrus fruits*).
- **Closed and Coherent Set:** The rows cannot be a random subset of a larger category unless a clear filter exists (e.g., ranking, time constraint).
- **Member Consistency:** All entities must logically belong to the group definition.
- **Specific Category:** The group must use precise and objective definitions; vague terms like "famous" or "important" are not allowed.
- **Natural Category:** The grouping must reflect a meaningful category that humans would reasonably recognize.
- **Context Usage:** Headers, captions, and nearby text may be used to refine the group definition when informative.
- **Conservative Validation:** If the grouping logic is unclear or ambiguous, the table should be rejected.


In [9]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(random.sample(cleaned_data, 1000))
await group_constructor.get_groups()

🚀 Processing 1000 tables...


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/1000 [00:00<?, ?it/s]

LLM Rejected,87
Entity Column has Numeric/Date Values,52
Duplicate Entities Found,152
Empty Entity Cells,3
Top-X Mismatch,1
Entity Column in Kill-List,4
Missing Header or Entity Column,1


In [18]:
group_constructor.display_debug_samples()

Island,Gaelic Name,Meaning of name,Location,Area ( ha ),Highest point,Last inhabited
Alloa Inch,Unknown,Alloa island,grid reference NS871915,c. 33,<5,20th century?
Bass Rock,Unknown,"Possibly from Gaelic bathais , meaning ""forehead"".",grid reference NT602874,7.5,107,1970s
Craigleith,Creag Lìte,Rock of Leith,grid reference NT555868,5,24,Inhabitation unlikely
Cramond Island,Unknown,island of the fort on the Almond river,grid reference NT194785,7.7,28,Possibly World War II
Fidra,Unknown,"From the Norse for ""feather island""",grid reference NT513868,10,20,1970
Inchcolm,Innis Choluim,Calum/Malcolm or Columba 's island,grid reference NT189824,9,34,First decade of the 21st century
Inchgarvie,Innis Garbhach,rough island,grid reference NT136795,0.83,19,Probably World War II
Inchkeith,Possibly Innis Cheith or Innis Coit .,wooded island,grid reference NT294825,20,59,1986
Inchmickery,Possibly Innis nam Bhiocaire,"Possibly ""island of the vicar""",grid reference NT207803,<5,14,Probably World War II
Lamb,Unknown,From a common Norse name for a small island,grid reference NT536867,0.5,15,Inhabitation unlikely


### Validate a small subset using Gemini 3.1 pro:

### RULES FOR VALIDITY

1. **FATEFULNESS:** Members must logically match the Group Name. Reject if the description is misleading, overly broad, or contains unrelated concepts.
2. **CLOSED_SET:** The group must be coherent and complete. Reject random subsets (e.g., "6 random cities in France"). 
   *Exception:* Specific filters (e.g., "Top 10", "Oldest 5", "Members in 2024") are valid.
3. **SPECIFICALITY:** The group must have a narrow, clear concept. Reject vague names like "Famous movies" or "Important countries."
4. **NATURALNESS:** The category must be one that humans would reasonably group together. Reject arbitrary rules (e.g., "Cities starting with Q that have 3 vowels").
5. **CORRECT_ENTITY:** The Entity Column must contain the primary entities, not metadata (Rank, Year, ID) or attributes.

In [11]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import GroupValidator

group_val = GroupValidator(random.sample(group_constructor.valid_tables, 200))
await group_val.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/200 [00:00<?, ?it/s]

"● CLOSED_SET 4 occurrences Wikipedia Reference ↗ Top 9 Argentine provinces or regions by Human Development Index (2021) The group represents a random or arbitrarily truncated subset of 9 Argentine provinces rather than a complete list of all 24 jurisdictions, and 'Top 9' is an unnatural cutoff that does not accurately reflect the actual highest HDI rankings. Entity Column: Province or region"
"● CORRECT_ENTITY 2 occurrences Wikipedia Reference ↗ Kings of Greece from the House of Schleswig-Holstein-Sonderburg-Gl3ccksburg The Entity Column contains a concatenation of the entity's name and their reign dates, rather than just the primary entity name. Entity Column: Name Reign"
"● FATEFULNESS 4 occurrences Wikipedia Reference ↗ Nashville Sounds players who won Southern League awards The group name specifies 'players', but the entity members include managers (Stump Merrill) and executives (Larry Schmittou, Bob Jamison), contradicting the group name. Entity Column: Recipient"
"● NATURALNESS 2 occurrences Wikipedia Reference ↗ Top 9 oldest buildings and structures in the Regional Municipality of Waterloo completed in the 1890s The combination of 'Top 9 oldest' and 'completed in the 1890s' forms an overly specific and unnatural grouping, likely an artifact of table extraction rather than a recognized category. Entity Column: Building"
"● SPECIFICALITY 1 occurrences Wikipedia Reference ↗ Well-known Turkic dialects The group name 'Well-known Turkic dialects' relies on the subjective term 'well-known', making the concept vague and not a well-defined category. Entity Column: Dialect"


## 4. Select Preferred Question Strategies for each Entity Group:

In [57]:
from knowledge_maze.create_dataset.create_questions.question_generation import StrategySelector

selector = StrategySelector(group_constructor.valid_tables)
selector.get_strategies()

Sanitizing tables:   0%|          | 0/700 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/700 [00:00<?, ?it/s]

Strategy Name,Selection Count
COUNT_TOTAL,367
EXTREME_TEMPORAL,318
UNIQUE_DIVERSITY,294
FILTER_COUNT_TEMPORAL,209
COMPARATIVE_DIFFERENCE,160
CATEGORICAL_FILTER_LOOKUP,158
FREQUENCY_MODE,152
TOP_K_RANKING,142
EXTREME_NUMERICAL,133
FILTER_COUNT_NUMERICAL,88


In [55]:
selector.debug_strategy_samples("CATEGORICAL_FILTER_LOOKUP")

Award,Work
BFI London Film Festival,The Falling
Cannes Film Festival,— N/a
Capri Hollywood International Film Festival,Oppenheimer
Dublin International Film Festival,Lady Macbeth
Festival 2 Cinéma 2 Valenciennes,Lady Macbeth
Montclair Film Festival,Lady Macbeth
Santa Barbara International Film Festival,—
Nominated Song,Category
""" Tum Hi Ho """,Male Vocalist of The Decade
""" Ae Dil Hai Mushkil """,Male Vocalist of The Decade


## 5. Create Complex Reasoning Questions w.r.t. Entity Groups & Selected Strategies:

In [133]:
from knowledge_maze.create_dataset.create_questions.question_generation import TableSanitizer, StrategySelector, QuestionGenerator

generator = QuestionGenerator(random.sample(selector.updated_data, 300), strategy_definitions=load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await generator.generate_questions()

🚀 Starting Question Generation for 300 tables...
--- Round 1: Processing 300 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/300 [00:00<?, ?it/s]

--- Round 2: Processing 11 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/11 [00:00<?, ?it/s]

--- Round 3: Processing 3 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/3 [00:00<?, ?it/s]

--- Round 4: Processing 1 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/1 [00:00<?, ?it/s]

Strategy Name,Success,Tried,Rate
COUNT_TOTAL,51,51,100%
FILTER_COUNT_TEMPORAL,34,34,100%
CATEGORICAL_FILTER_LOOKUP,34,34,100%
EXTREME_TEMPORAL,33,35,94%
UNIQUE_DIVERSITY,32,35,91%
FREQUENCY_MODE,31,33,94%
EXTREME_NUMERICAL,18,18,100%
TOP_K_RANKING,17,19,89%
COMPARATIVE_DIFFERENCE,17,21,81%
FILTER_COUNT_NUMERICAL,17,18,94%


In [145]:
from knowledge_maze.create_dataset.create_questions.question_generation import QuestionFilter

val = QuestionFilter(generator.valid_results)
val.run_pipeline()

Starting post-processing with 299 samples...


Rule,Definition,Count
DUPLICATE_DATA,Question is too similar to another question in the dataset.,5
MISSING_COLUMNS,Table lacks an attribute column for this strategy.,2
MISSING_QUESTION,Question field was empty or null.,0
MISSING_TABLE,No table data was found for this sample.,0
EXTRA_COLUMNS,Table contains more than one attribute column (schema violation).,0
EMPTY_DATA,Attribute column exists but has no data rows.,0


## 6. Filter Out Bad Samples Using a Validator Model:

In [146]:
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator
validator = QuestionValidator(random.sample(val.approved_data, 50), load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await validator.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/50 [00:00<?, ?it/s]

## 7. Data Statistics:

1. model based accuracy
2. human based accuracy
3. subject distribution
4. number of evidences distribution
5. strategy distribution (2-layered pie chart)

## API Stuff:

In [199]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import GroupValidatorTest

group_val = GroupValidatorTest(random.sample(group_constructor.valid_tables, 4))
await group_val.validate()

rev = Counter([sample['verdict'] for sample in group_val.output_data])
print(f"{100 * rev[True] / sum(rev.values()):.1f}%")
rev

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/4 [00:00<?, ?it/s]

ChatCompletion(id='gen-1773190353-q2hzZ6AWwTuuDHm1Vo26', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='True', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, reasoning='**Evaluating Group Cohesion**\n\nI\'m currently focused on the coherence of the group "Wind power stations in Iran." Initial assessments suggest a common theme, but I need to ascertain if the group is specific enough to be meaningful. I\'m investigating if the shared location and function constitute a naturally occurring group, or if further refinement is necessary. The group size seems manageable for now.\n\n\n**Defining Group Membership**\n\nI\'ve confirmed each entity\'s presence within the specified geographic and functional criteria. The group\'s coherence is demonstrable, with all members adhering to the definition of "wind power stations in Iran." The entity column\'s members are valid, and the group definition provides a clea

Counter({None: 4})

In [184]:
for s in [sample for sample in group_val.output_data if not sample['verdict']]:
    print(s['Group name'])
    print(s['Entity column'])
    print(s['Wiki url'])
    print("-"*100)

Members of Task Force ✱/Suicide Squad Dark
Character
https://en.wikipedia.org/wiki/List_of_Suicide_Squad_members#Task_Force_%E2%9C%B1/Suicide_Squad_Dark
----------------------------------------------------------------------------------------------------
Top 5 mainland Chinese albums by annual sales percentage on the Billboard China V Chart in 2016
Album
https://en.wikipedia.org/wiki/List_of_Billboard_China_V_Chart_top_10_albums_of_2016#Mainland_%28Domestic%29_albums
----------------------------------------------------------------------------------------------------
Taiwanese films released in 2001
Title
https://en.wikipedia.org/wiki/List_of_Taiwanese_films_of_the_2000s#2001
----------------------------------------------------------------------------------------------------


In [187]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import GroupValidator

group_val = GroupValidator([sample for sample in group_val.output_data if sample['verdict']])
await group_val.validate()

rev = Counter([sample['verdict'] for sample in group_val.output_data])
print(f"{100 * rev['ACCEPT'] / sum(rev.values()):.1f}%")
rev

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/17 [00:00<?, ?it/s]

Input tokens: 642
Output tokens: 188
Input tokens: 577
Output tokens: 271
Input tokens: 578
Output tokens: 207
Input tokens: 519
Output tokens: 255
Input tokens: 629
Output tokens: 236
Input tokens: 622
Output tokens: 251
Input tokens: 533
Output tokens: 369
Input tokens: 624
Output tokens: 406
Input tokens: 616
Output tokens: 430
Input tokens: 606
Output tokens: 548
Input tokens: 630
Output tokens: 182
Input tokens: 596
Output tokens: 217
Input tokens: 607
Output tokens: 155
Input tokens: 571
Output tokens: 238
Input tokens: 544
Output tokens: 219
Input tokens: 580
Output tokens: 892
Input tokens: 581
Output tokens: 181
94.1%


Counter({'ACCEPT': 16, 'CLOSED_SET': 1})